In [26]:
import shop_agent as sa

In [37]:
print('today :', sa.TODAY)
print('db    :', sa.DB)
for k, v in sa.POLICY.items():
    print(f'{k:<35} {v}')

today : 2026-07-26
db    : C:\Work Stuff\Personal Project 2026\ResaleResolutionAgent\data\shop.db
dataset_today                       2026-07-26
return_window_days                  30
auto_refund_limit_gbp               100
condition_dispute_partial_pct       25
condition_dispute_limit_gbp         50
hygiene_excluded_categories         ['swimwear']
authenticity_claims                 always_escalate
shop_fault_overrides_final_sale     True


In [39]:
for t in sa.SHOP_TOOL_DEFS:
    print(f'[{t.name}]')
    print(t.description)
    print()


[get_customer]
Fetch a customer record by ID. Return name,email,account status ('active' or 'suspended'), and joined_date. Use for account-level context,such as confirming the account is in good standing before a refund.Does NOT return orders - use lookup_order for anything order-specific.

[lookup_order]
Look for customer order by ID. Return order_date,delivery_date,status,days_since_delivery, in_return_window,already_refund, customer_id anditem details(brand, category, condition, price_gbp, final_sale)Call this before deciding on any refund, or exchange - it is the single source of truth for eligibility facts.Do NOT process anything

[process_refund]
Issue a refund against an order.Use ONLY when lookup_order confirms the order is refundable and the amount is below the GBP 100 auto-refund limit. Pass reason='condition_partial' for a goodwill partial refund, and reason='shop_fault' when the shop is at fault(wrong item sent, damaged in transit, misdescribed) - shop_fault is the only rea

In [ ]:
#raw loop
with sa.sandbox():
    calls, reply, stop = await sa.run_case_raw(
        "Hi, I'd like to return the wrap dress from order ORD-001. "
        "It doesn't suit me. Can I get a refund?")

for name, inp in calls:
    print(name, inp)

print("\nstop_reason:",stop)
print("---\n", reply)

lookup_order {'order_id': 'ORD-001'}
process_refund {'order_id': 'ORD-001', 'amount_gbp': 35, 'reason': 'return_window'}

stop_reason: end_turn
---
 Yes — your refund of £35 is approved and processed.

The Whistles wrap dress was delivered 5 days ago, so it's comfortably inside our 30-day return window, and it wasn't a final-sale item. The £35 will go back to your original payment method, usually within 3–5 working days depending on your bank.

Just send the dress back to us in the condition it arrived. If you need anything else, I'm here.


In [43]:
#SKD
with sa.sandbox():
    calls, reply = sa.run_sync(sa.run_case_sdk(
        "Hi, I'd like to return the wrap dress from order ORD-001."
        "It doesn't suit me. Can I get a refund?"))

for name, inp in calls:
    print(name, inp)
print("\nstop_reason:",stop)
print("\n---\n", reply)

mcp__shop__lookup_order {'order_id': 'ORD-001'}

stop_reason: end_turn

---
 I can't issue a refund on this one — it's already been refunded.

Our records show ORD-001 (Whistles wrap dress, £35) was delivered on 21 July and a full refund has already been processed against it. The order is closed out on our side, so there's nothing further to pay back.

A couple of things worth saying:

- If that refund hasn't landed in your account yet, refunds usually take 3–5 working days to show after they're issued. Let me know if it's been longer and I'll get someone to trace it.
- If you still have the dress and this is actually a second, separate item you're trying to return, tell me which order it came on and I'll check that one properly.

And just so you know for future: we're one-of-one on stock, so we can never do exchanges — it's always a refund instead.
